# Day 5 — Delta DML: MERGE, UPDATE, DELETE, SCD patterns

Plan for about 1.5–2 hours. You will build a small `dim_routes` table on ABFS, merge updates from the Python API and from a staging Delta path, run conditional merges, use UPDATE and DELETE (including no-op examples), review SCD Type 1 vs a simplified Type 2 layout, and try an optional SQL MERGE plus follow-up challenges.

SQL uses `delta.` paths where shown. Earlier notebooks in this course use the same ABFS layout and mount helper.

In [ ]:
%run ./02-Mount-Azure-Data-Lake

In [ ]:
from pyspark.sql.functions import col, concat_ws, current_date, current_timestamp, lit, sha2, when

BASE_PATH = base_path
DAY5 = BASE_PATH + "/day5"
P_DIM = DAY5 + "/delta/dim_routes"
P_SCD2 = DAY5 + "/delta/dim_routes_scd2"
SOURCE_CSV = BASE_PATH + "/2010-summary.csv"
print(P_DIM)

## S1 — Build **`dim_routes`** (surrogate key = `route_key`)

In [ ]:
from pyspark.sql.functions import coalesce, col, current_timestamp, lit

src = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(SOURCE_CSV)
    .select("DEST_COUNTRY_NAME", "ORIGIN_COUNTRY_NAME", "count")
    .withColumnRenamed("count", "flight_count")
    .withColumn("route_key", sha2(concat_ws("|", col("DEST_COUNTRY_NAME"), col("ORIGIN_COUNTRY_NAME")), 256))
    .withColumn("effective_from", current_date())
    .withColumn("updated_at", current_timestamp())
    .withColumn("source_path", coalesce(col("_metadata.file_path"), lit(SOURCE_CSV)))
)
src.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(P_DIM)
src.show(5, truncate=False)
print("rows", src.count())

## S2 — **MERGE** — apply corrections from staging

In [ ]:
from delta.tables import DeltaTable

dim = DeltaTable.forPath(spark, P_DIM)

updates = (
    spark.read.format("delta")
    .load(P_DIM)
    .filter(col("DEST_COUNTRY_NAME") == "United States")
    .limit(5)
    .withColumn("flight_count", col("flight_count") + lit(100))
    .withColumn("updated_at", current_timestamp())
)

(
    dim.alias("t")
    .merge(updates.alias("s"), "t.route_key = s.route_key")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

DeltaTable.forPath(spark, P_DIM).history(5).select("version", "operation", "operationMetrics").show(truncate=False)

## S2b — **Conditional** `whenMatchedUpdate` (only if staging is “newer”)

In [ ]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col, current_timestamp, lit

dim = DeltaTable.forPath(spark, P_DIM)
bump = (
    spark.read.format("delta")
    .load(P_DIM)
    .where("DEST_COUNTRY_NAME = 'Germany'")
    .limit(4)
    .withColumn("flight_count", col("flight_count") + lit(50))
    .withColumn("updated_at", current_timestamp())
)
(
    dim.alias("t")
    .merge(bump.alias("s"), "t.route_key = s.route_key")
    .whenMatchedUpdate(
        condition="s.flight_count > t.flight_count",
        set={"flight_count": "s.flight_count", "updated_at": "s.updated_at"},
    )
    .execute()
)
print("Conditional merge applied where staged flight_count exceeds target")

## S2c — **Stage → merge** (separate Delta path, full refresh pattern)

In [ ]:
from pyspark.sql.functions import coalesce, col, concat_ws, current_date, current_timestamp, lit, sha2

P_STAGE = DAY5 + "/delta/stage_routes_snapshot"
st = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(SOURCE_CSV)
    .select("DEST_COUNTRY_NAME", "ORIGIN_COUNTRY_NAME", "count")
    .withColumnRenamed("count", "flight_count")
    .withColumn("route_key", sha2(concat_ws("|", col("DEST_COUNTRY_NAME"), col("ORIGIN_COUNTRY_NAME")), 256))
    .withColumn("effective_from", current_date())
    .withColumn("updated_at", current_timestamp())
    .withColumn("source_path", coalesce(col("_metadata.file_path"), lit(SOURCE_CSV)))
    .limit(200)
)
st.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(P_STAGE)
print("Staging rows:", st.count(), "at", P_STAGE)

In [ ]:
from delta.tables import DeltaTable

dim = DeltaTable.forPath(spark, P_DIM)
stg = spark.read.format("delta").load(P_STAGE)
(
    dim.alias("t")
    .merge(stg.alias("s"), "t.route_key = s.route_key")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)
print("Merged staging snapshot into dim_routes")

## S3 — Verify merge on touched **route_key**s

In [ ]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col

keys = [r["route_key"] for r in spark.read.format("delta").load(P_DIM).where("DEST_COUNTRY_NAME = 'United States'").limit(5).select("route_key").collect()]
ver = int(DeltaTable.forPath(spark, P_DIM).history(1).select("version").collect()[0]["version"])
print("latest version", ver)
if ver >= 1:
    spark.read.format("delta").option("versionAsOf", ver - 1).load(P_DIM).filter(col("route_key").isin(keys)).select("route_key", "flight_count").orderBy("route_key").show(truncate=False)
spark.read.format("delta").load(P_DIM).filter(col("route_key").isin(keys)).select("route_key", "flight_count").orderBy("route_key").show(truncate=False)

## S4 — **`UPDATE`** (DeltaTable API — avoids SQL dialect quirks)

In [ ]:
from delta.tables import DeltaTable

DeltaTable.forPath(spark, P_DIM).update(
    condition="DEST_COUNTRY_NAME = 'Canada'",
    set={"flight_count": "flight_count + 1"},
)
print("Updated Canada rows (+1 flight_count each)")

**Alternative (SQL warehouse):** register a temp view backed by Delta, then `UPDATE` the view if your runtime supports it.

```python
# spark.read.format("delta").load(P_DIM).createOrReplaceTempView("dim_routes")
# spark.sql("UPDATE dim_routes SET flight_count = flight_count + 1 WHERE DEST_COUNTRY_NAME = 'Germany'")
```

## S5 — **`DELETE`** (SQL + **DeltaTable** API)

In [ ]:
# Safe no-op: predicate matches nothing
spark.sql(f"DELETE FROM delta.`{P_DIM}` WHERE DEST_COUNTRY_NAME = '__no_such_country__'")
print("SQL delete completed (0 rows if predicate empty)")

In [ ]:
from delta.tables import DeltaTable

# API delete — same semantics as SQL DELETE with predicate
DeltaTable.forPath(spark, P_DIM).delete("DEST_COUNTRY_NAME = '__also_missing__'")
print("DeltaTable.delete no-op demo OK")

## S5b — DELETE with a real predicate (optional; destructive)

**Warning:** deletes real rows. Pick a **low-value** country present in your CSV (e.g. a tiny country) **or** skip.

```python
# Example ONLY after backup / clone:
# DeltaTable.forPath(spark, P_DIM).delete("DEST_COUNTRY_NAME = 'SomeTinyCountry'")
```

## S6 — **SCD Type 1** (current truth only) — recap

**Type 1:** Overwrite attributes in place — **no history** of old values. Your **`MERGE` + `whenMatchedUpdateAll`** above is Type 1 for `flight_count` / `updated_at`.

## S7 — **SCD Type 2** — keep history (simplified pattern)

In [ ]:
from pyspark.sql.functions import col, current_date, lit, row_number
from pyspark.sql.window import Window

# Drop Type-1 audit cols; keep business attributes for SCD2 toy layout
base = spark.read.format("delta").load(P_DIM).drop("effective_from", "updated_at", "source_path").withColumnRenamed("flight_count", "metric_value")
# Build SCD2-style snapshot with is_current + effective dates (demo only)
w = Window.partitionBy("route_key").orderBy(col("metric_value").desc())
scd2 = (
    base.withColumn("effective_start", current_date())
    .withColumn("effective_end", lit(None).cast("date"))
    .withColumn("is_current", lit(True))
    .withColumn("version_num", row_number().over(w))
)
scd2.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(P_SCD2)
scd2.show(5, truncate=False)

**Production SCD2** usually closes old rows (`effective_end`, `is_current=false`) in the same **`MERGE`** using `whenMatchedUpdate` + `whenNotMatchedInsert`. This cell is a **toy** layout for teaching.

## S8 — Extra **MERGE** scenarios (patterns)

- **Partial update:** `whenMatchedUpdate(set = { "flight_count": "s.flight_count", "updated_at": "s.updated_at" })` — not `updateAll`.

- **Delete when missing in source:** `whenNotMatchedBySourceDelete` (Delta) for slowly syncing full snapshots.

- **Dedupe keys:** always validate **one row per business key** before merge.

## S9 — SQL **MERGE** template (Databricks)

```sql
-- MERGE INTO delta.`abfss://.../dim` AS t
-- USING updates_view AS s
-- ON t.route_key = s.route_key
-- WHEN MATCHED AND s.updated_at > t.updated_at THEN UPDATE SET *
-- WHEN NOT MATCHED THEN INSERT *
```

## S10 — **Idempotent** batch load (pattern)

1. Stage data in **temp** Delta or view.  
2. **`MERGE`** into target on **natural/surrogate key**.  
3. Optionally **`DELETE`** targets not in stage (full refresh).  
4. Log **`operationMetrics`** from history for monitoring.

## S11 — Six quick drills

- Write a **MERGE** that **only** updates when `s.flight_count <> t.flight_count`.

- Use **`spark.sql`** `DESCRIBE DETAIL` on **`P_DIM`** after each DML block.

- Count versions in **`DESCRIBE HISTORY`** for **`P_DIM`** — map to operations you ran.

- Create a **temp view** on **`P_DIM`** and run **`SELECT ... EXCEPT`** against a snapshot `versionAsOf`.

- Explain when **`UPDATE`** is clearer than **`MERGE`** (single-table fix vs join upsert).

- List **risks** of **`DELETE`** without **merge** predicates in production.

## S12 — **`MERGE` in SQL** (same semantics as Python API; try/except)

In [ ]:
from pyspark.sql.functions import col, current_timestamp, lit
from delta.tables import DeltaTable

# Tiny bump for Germany — register as temp view, merge via SQL
bump_sql = (
    spark.read.format("delta")
    .load(P_DIM)
    .where("DEST_COUNTRY_NAME = 'Germany'")
    .limit(3)
    .withColumn("flight_count", col("flight_count") + lit(7))
    .withColumn("updated_at", current_timestamp())
)
bump_sql.createOrReplaceTempView("sql_merge_src")
try:
    spark.sql(
        f"MERGE INTO delta.`{P_DIM}` AS t USING sql_merge_src AS s ON t.route_key = s.route_key "
        "WHEN MATCHED THEN UPDATE SET *"
    )
    print("SQL MERGE OK")
except Exception as e:
    print("SQL MERGE:", type(e).__name__, str(e)[:200])
DeltaTable.forPath(spark, P_DIM).history(3).select("version", "operation", "operationMetrics").show(truncate=False)

## S12b — Parse **`operationMetrics`** from last **MERGE** (Python)

In [ ]:
from delta.tables import DeltaTable

h = DeltaTable.forPath(spark, P_DIM).history(1)
row = h.collect()[0]
print("Last operation:", row["operation"])
print("Metrics:", row["operationMetrics"])

## S13 — Challenges

Work in the cells below. Review with the group or in a follow-up session.

### K1 — Write a **MERGE** that **inserts** when `route_key` is new and **updates only** `flight_count` + `updated_at` when matched (no `updateAll`).

In [ ]:
# Your answer / code:


### K2 — Explain **`whenNotMatchedBySourceDelete`** — when would you use it for a **full snapshot** sync?

In [ ]:
# Your answer / code:


### K3 — After your MERGEs, how many **versions** does **`DESCRIBE HISTORY`** show for **`P_DIM`**? List **operation** names in order.

In [ ]:
# Your answer / code:


### K4 — **One paragraph:** Why is **SCD Type 2** harder to implement correctly than **Type 1** in a streaming pipeline?

In [ ]:
# Your answer / code:


## End of notebook 04

This finishes the Day 5 Delta exercises on ABFS (same `%run` mount pattern as earlier days). Further topics such as streaming reads of Delta are covered elsewhere in the course if applicable.